In [19]:
import os 
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["MISTRAL_API_KEY"] = 'Eajkd7toYyYCEoU1LQiNFcPTvyK3ONep'
os.environ["LANGSMITH_PROJECT"] = "llm_rag_rerank"

Aim - Create a RAG based LLM to fetch accurate information
Steps - classify query, get context based on the question, generate answer, evaluate the answer

In [20]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document

In [21]:
## store related document in vectorstore
doc_loader = PyPDFLoader("transformers.pdf")
doc = doc_loader.load()

doc_splitter = RecursiveCharacterTextSplitter()
doc_chunk = doc_splitter.split_documents(doc)

embedding = MistralAIEmbeddings()
vector_store = FAISS.from_documents(doc_chunk, embedding)
retriever = vector_store.as_retriever(search_kwargs={"k":2})

/Users/ranishreedey/Library/Python/3.9/lib/python/site-packages/langchain_mistralai/embeddings.py:181: UserWarning: Could not download mistral tokenizer from Huggingface for calculating batch sizes. Set a Huggingface token via the HF_TOKEN environment variable to download the real tokenizer. Falling back to a dummy tokenizer that uses `len()`.
  warnings.warn(


In [22]:
class State(TypedDict):
    query : str
    query_type : str
    context : str
    refined_context : str
    answer : str
    score : str

In [23]:
def classify_query(state : State):
    """Classifies a user query as a generic query or a specific topic related.
    In this case the Transformer Model"""
    
    llm = ChatMistralAI(model = "mistral-large-latest")
    system_prompt = """You are an expert in classifying a given query as - generic or transformer-related.
                        If the query is anything about the Transformer Model, then classify it as 'transformer-related' else 'generic'.
                    """

    query_type = llm.invoke(input = [("system",system_prompt),
                                   ("human",state["query"])]).content
    
    return {"query_type" : query_type, "query" : state["query"], "context": "", "refined_context": "", "answer":"", "score" : ""}

In [24]:
def get_context(state : State):
    """ Gets the Context from the internal document using RAG"""
    context = retriever.invoke(input=state["query"])

    return { "query_type" : state["query_type"], "query" : state["query"], "context": context, "refined_context": "", "answer":"", "score" : ""}


In [25]:
def re_rank_context(state : State):
    prompt = f"""Based on the given query and context-list, re-rank the contexts based on the relavancy to the query.
    <query>
    {state["query"]}

    <list of contexts>
    {state['context']}
    """
    llm = ChatMistralAI(model = "mistral-large-latest")
    response = llm.invoke(input = [("system",prompt),
                                    ("human",state["query"])]).content
    return {"query_type" : state["query_type"], "query" : state["query"], "context": state["context"], "refined_context" : response, "answer":"", "score" : ""}
    

In [26]:
def generate(state : State):
    """Generate the answer based on the given input and context"""
    prompt = f"""With the given context answer the query.
                    ALWAYS append your answer with 'USING RAG'

                    <context>
                    {state["context"]}
                    </context>
                
                    query : {state["query"]}"""
    
    llm = ChatMistralAI(model = "mistral-large-latest", max_tokens = 100)
    response = llm.invoke(input = [("system",prompt),
                                    ("human",state["query"])]).content
    return {"query_type" : state["query_type"], "query" : state["query"], "context": state["context"],  "refined_context" : state["refined_context"], "answer":response, "score" : ""}
    # return {"query" : state["query"], "answer":response}
    # return query, answer


In [27]:
def evaluate(state : State):
    """Evaluate the answer using another LLM as an evaluator"""
    system_prompt = f"""With the given query and answer, rate the answer from 1-5. 5 is the highest score.
                    {state["query"]}
                    {state["answer"]}
                    """
    llm = ChatMistralAI(model='mistral-large-latest')
    score = llm.invoke(input = [("system",system_prompt),
                                   ("human",f"""{state["query"]}
                                    {state["answer"]}""")])
    return {"query_type" : state["query_type"], "query" : state["query"], "context": state["context"], "refined_context" : state["refined_context"], "answer": state["answer"], "score" : score.content}


In [28]:
def router(state : State):
    """Route to the next step. In case of generic query simply answer and avoid fetching context"""
    if 'transformer-related' in state["query_type"].lower():
        return "get_context"
    else:
        return END        

In [29]:
## Initialize the graph to orchestrate the steps
graph = StateGraph(State)
graph.add_node("classify_query", classify_query)
graph.add_node("get_context", get_context)
graph.add_node("re_rank_context",re_rank_context)
graph.add_node("generate",generate)
graph.add_node("evaluate", evaluate)

In [30]:
## Add edges
graph.add_edge(START,"classify_query")
graph.add_conditional_edges("classify_query",router)
graph.add_edge("get_context","re_rank_context")
graph.add_edge("re_rank_context", "generate")
graph.add_edge("generate","evaluate")
graph.add_edge("evaluate", END)

graph.set_entry_point("classify_query")

In [31]:
## compile the graph
workflow = graph.compile()
workflow.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnablePassthrough(), metadata=None), 'classify_query': Node(id='classify_query', name='classify_query', data=classify_query(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 'get_context': Node(id='get_context', name='get_context', data=get_context(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 're_rank_context': Node(id='re_rank_context', name='re_rank_context', data=re_rank_context(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 'generate': Node(id='generate', name='generate', data=generate(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 'evaluate': Node(id='evaluate', name='evaluate', data=evaluate(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={})

In [32]:
## infer/invoke the graph
result = workflow.invoke({"query":"what is multi head attention in transformers?"})
result

{'query': 'what is multi head attention in transformers?',
 'query_type': 'Based on the query, "what is multi head attention in transformers?", the classification would be:\n\n**transformer-related**\n\nThe query specifically mentions "multi head attention" and "transformers," which are key components and concepts related to the Transformer model architecture.',
 'context': [Document(id='9e0ee7c8-a900-4eef-9f10-7b00e49a4b49', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-02-09T02:33:09+00:00', 'author': '', 'keywords': '', 'moddate': '2024-02-09T02:33:09+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'transformers.pdf', 'total_pages': 10, 'page': 3, 'page_label': '3'}, page_content='the K×D matrices Uq and Uk are the only parameters of this mechanism.10\nMulti-head self-attention (MHSA). In the self-attention 